### Notebook to combine the previously estimated zero-points with forced photometry

This Notebook outputs zero-point-corrected photometry (one .csv file per MJD). It also outputs comparison plots of the old and new photometry.


In [ ]:

""" loading the relevant packages """

from glob import glob
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as tck
import matplotlib
import matplotlib.cm as cm

# setting figure style
plt.rcParams.update({
    "text.usetex" : False,
    "font.family" : "Arial", 
    "mathtext.fontset" : "stix",
    "font.size" : 14
})

fontsize = plt.rcParams["font.size"]


In [ ]:

""" defining filepaths for offset values and forced phot file """

offsets_filepath = "Bundles/CMFs/offsets.txt"

forced_phot_filepath = "Bundles/forced_phot_2026-05-05.txt"



### Defining some useful functions



In [ ]:

def _mkdir(newdir):
    """ Creates the required directory for the output files. """
    """works the way a good mkdir should :)
        - already exists, silently complete
        - regular file in the way, raise an exception
        - parent directory(ies) does not exist, make them as well
    """
    if os.path.isdir(newdir):
        pass
    elif os.path.isfile(newdir):
        raise OSError("a file with the same name as the desired " \
                      "dir, '%s', already exists." % newdir)
    else:
        head, tail = os.path.split(newdir)
        if head and not os.path.isdir(head):
            _mkdir(head)
        #print "_mkdir %s" % repr(newdir)
        if tail:
            os.mkdir(newdir)
    return newdir


In [ ]:

def fnu2ABmag(flux):
    # simple function to convert flux --> AB mag
    mag = -2.5 * np.log10(flux) + 23.90 # flux in micro-Jy
    return mag


def ABmag2fnu(mag):
    # simple function to convert AB mag --> flux
    flux = 10**((mag - 23.90) / (-2.5))
    return flux


def efnu2eABmag(flux, eflux):
    # simple function to convert flux error --> AB mag error
    emag = (2.5 / np.log(10)) * (eflux / flux)
    return emag


In [ ]:

def merge_dfs_fn(ZP_df, forced_phot_df):
    """ function to merge two DataFrames with approximately
    matching MJD values """

    # Round to reduce precision noise if needed
    ZP_df["mjd"] = ZP_df["mjd"].round(5)
    forced_phot_df["mjd"] = forced_phot_df["mjd"].round(5)

    # Sort by mjd for merge_asof
    ZP_df = ZP_df.sort_values("mjd")
    forced_phot_df = forced_phot_df.sort_values("mjd")

    # Merge with tolerance of 0.002
    merged_df = pd.merge_asof(
        ZP_df,
        forced_phot_df,
        on="mjd",
        by="filter",  # ensures same filter is used
        direction="nearest",
        tolerance=0.002,
    )

    # now we keep only the relevant columns
    merged_df = merged_df[["mjd", "filter", "exptime", "ujy", "dujy", "offset_median", "cal_psf_mag", "psf_inst_mag_sig"]]

    # computing the magnitudes from the forced flux values
    # these should match (exactly?) with the "cal_psf_mag" and "psf_inst_mag_sig" values
    merged_df["Mag(AB)"] = fnu2ABmag(merged_df["ujy"])
    merged_df["eMag(AB)"] = efnu2eABmag(merged_df["ujy"], merged_df["dujy"])

    return merged_df


In [ ]:

def specific_MJD_df_fn(merged_df, mjd_of_interest):
    """ function to cull merged_df, retaining only photometry from the epoch of interest """

    # the +/- MJD window within which to consider the MJD to be "now"
    mjd_offset = 0.3

    indexes = merged_df[(merged_df["mjd"] < mjd_of_interest - mjd_offset) | (merged_df["mjd"] > mjd_of_interest + mjd_offset)].index

    merged_df.drop(indexes, inplace=True)

    merged_df.sort_values(by="mjd", inplace=True)
    merged_df.reset_index(drop=True, inplace=True)

    return merged_df


In [ ]:

def plotter_fn(new_df, merged_df, mjd_of_interest):
    """ Function to generate the plot comparing un-corrected and corrected photometry. """

    fig = plt.figure(figsize=(16, 8))

    ax = plt.subplot()
    ax.tick_params(axis='both', direction='in', top=True, right=True, left=True, bottom=True, which='both')
    ax.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax.yaxis.set_minor_locator(tck.AutoMinorLocator())

    colorlist = cm.jet(np.linspace(0.1, 0.9, 6))
    colorlist_counter = 0

    color_dict = dict(
        {
            'g' : colorlist[0],
            'r' : colorlist[1],
            'i' : colorlist[2],
            'z' : colorlist[3],
            'y' : colorlist[4],
            'w' : colorlist[5],
        }
    )

    offset = 20

    for aa in range(len(new_df)):

        tmp_filter = new_df['filter'][aa]
        tmp_lambda = new_df['filter_centre(AA)'][aa]
        tmp_width = new_df['filter_width(AA)'][aa] / 2
        tmp_flam = new_df['Mag(AB)'][aa]
        tmp_flam_err = new_df['eMag(AB)'][aa]

        marker='o'

        tmp_label = (f'{tmp_filter}')

        plt.plot(tmp_lambda, tmp_flam, color=color_dict[tmp_filter], marker=marker, label=tmp_label, zorder=99, markersize=8, linestyle='')
        plt.errorbar(tmp_lambda, tmp_flam, xerr=tmp_width, yerr=tmp_flam_err, ecolor=color_dict[tmp_filter], elinewidth=2, capsize=3, capthick=2, linestyle='', zorder=99)
        colorlist_counter += 1

    colorlist_counter = 0

    color="black"
    counter = 1

    for bb in range(len(merged_df)):

        if counter == 1:
            label = "Un-corrected"
        else:
            label = ""

        plt.plot(merged_df["centre"][bb]-offset, float(merged_df["cal_psf_mag"][bb]), color=color, linestyle="", marker="o", label=label, alpha=0.7)
        plt.errorbar(merged_df["centre"][bb]-offset, float(merged_df["cal_psf_mag"][bb]), xerr=merged_df["width"][bb]/2, yerr=merged_df["psf_inst_mag_sig"][bb], linestyle="", color=color, alpha=0.7, elinewidth=2, capsize=3, capthick=2)

        counter += 1


    plt.legend(frameon=False, loc="best", ncol=2)
    plt.title(
        "Pan-STARRS forced photometry before and after zero-point correction. \n"
        "The final corrected photometry is coloured, while the `default' photometry is plotted in black. \n"
        f"$\sim$ MJD {mjd_of_interest}"
    )

    plt.gca().invert_yaxis()

    plt.xlabel("Observed wavelength + offset ($\mathrm{\AA}$)")
    plt.ylabel("Apparent magnitude (AB mag)")

    plt.savefig(f"{_mkdir('Photometry/pdf_files')}/photometry_comparison_MJD~{mjd_of_interest}.pdf", dpi=900, bbox_inches="tight")

    plt.close()

    return 



In [ ]:

""" defining Pan-STARRS filter information """

PS_filters           = ["g",     "r",     "i",     "z",     "y",     "w"]
PS_filter_eff_centre = [4810.16, 6155.47, 7503.03, 8668.36, 9613.60, 5980.70]
PS_filter_eff_width  = [1053.08, 1252.41, 1206.62, 997.72,  638.98,  2561.73]

PS_filter_df = pd.DataFrame(
    {
        "Filter" : PS_filters,
        "Eff_centre" : PS_filter_eff_centre,
        "Eff_width" : PS_filter_eff_width,
    }
)


In [ ]:

""" reading in the offsets.txt file """

ZP_df = pd.read_csv(offsets_filepath, sep=",")
ZP_df


In [ ]:

""" reading in the forced photometry file downloaded from the Pan-STARRS page """

forced_phot_df = pd.read_csv(forced_phot_filepath, sep="\s+")

forced_phot_df = forced_phot_df.rename(columns={"#mjd" : "mjd"})
forced_phot_df



### [OPTIONAL] Few cells below here for optional data curation



In [ ]:

# """ Truncating entries before SN """

# mjd_start = 60880

# indexes = forced_phot_df[forced_phot_df["mjd"] < mjd_start].index

# forced_phot_df.drop(index=indexes, inplace=True)
# forced_phot_df.reset_index(drop=True, inplace=True)

# forced_phot_df



In [ ]:

# """ Setting negative forced phot values to 0 """

# indexes = forced_phot_df[forced_phot_df["ujy"] < 0].index

# forced_phot_df.loc[indexes, "ujy"] = 1e-3

# print(forced_phot_df.iloc[indexes])

# forced_phot_df


In [ ]:

""" getting list of all MJDs we observed """

unique_MJDs = ZP_df['mjd'].astype(int).unique()
print("Unique MJD values (as integers): \n", unique_MJDs)

# set sub MJD value; i.e., when on MJD XXXXX.xxx were the OBs typically taken.
# here we are setting the .xxx value here
sub_MJD = 0.5


In [ ]:

""" running the main loop """

for MJD in unique_MJDs:

    mjd_of_interest = MJD + sub_MJD

    merged_df = merge_dfs_fn(ZP_df, forced_phot_df)

    merged_df = specific_MJD_df_fn(merged_df, mjd_of_interest)

    # now we correct the forced mag values for the ZP correction
    merged_df["Corrected_Mag(AB)"] = merged_df["Mag(AB)"] - merged_df["offset_median"]

    # needed to properly average multiple exposures
    merged_df["Corrected_flux(uJy)"] = ABmag2fnu(merged_df["Corrected_Mag(AB)"])

    ##################################################################################################
    # now we compute the average mag and mag error for the multiple exposures
    ##################################################################################################


    # defining a smaller, tidier df to store the final data in
    new_df = pd.DataFrame(columns=merged_df.columns)
    new_df = pd.DataFrame(columns=["mjd", "filter", "filter_centre(AA)", "filter_width(AA)", "TotalExptime(s)", "Mag(AB)", "eMag(AB)", "Flux(uJy)", "eFlux(uJy)"])

    ##################################################################################################
    # building the final df with averaged mag values (where relevant)
    ##################################################################################################


    counter = 0

    for filter in ["g", "r", "i", "z", "y", "w"]:

        tmp_df = merged_df[merged_df["filter"] == filter].reset_index(drop=True)

        centre = PS_filter_df[PS_filter_df['Filter'] == filter]["Eff_centre"].values[0]
        width = PS_filter_df[PS_filter_df['Filter'] == filter]["Eff_width"].values[0]

        if len(tmp_df) == 1:

            new_df.loc[counter] = [
                tmp_df["mjd"].values[0],
                tmp_df["filter"].values[0],
                centre,
                width,
                tmp_df["exptime"].values[0],
                tmp_df["Corrected_Mag(AB)"].values[0],
                tmp_df["eMag(AB)"].values[0],
                tmp_df["Corrected_flux(uJy)"].values[0],
                tmp_df["dujy"].values[0],
            ]

        elif len(tmp_df) > 1:

            av_flux = np.mean(ABmag2fnu(tmp_df["Corrected_Mag(AB)"]))
            av_mag = fnu2ABmag(av_flux)

            av_eflux = np.sqrt(np.sum(tmp_df["dujy"]**2)) / len(tmp_df)
            av_emag = efnu2eABmag(av_flux, av_eflux)
            
            av_mjd = np.mean(tmp_df["mjd"])

            total_exptime = np.sum(tmp_df["exptime"])

            new_df.loc[counter] = [
                av_mjd,
                filter,
                centre,
                width,
                total_exptime,
                av_mag,
                av_emag,
                av_flux,
                av_eflux,
            ]

        counter += 1

    new_df.reset_index(inplace=True, drop=True)

    ##################################################################################################
    # computing the relevant filter bits for the uncorrected photometry - needed for plotting reasons below...
    ##################################################################################################


    centres = []
    widths = []

    for aa in range(len(merged_df)):
        centre = PS_filter_df[PS_filter_df['Filter'] == merged_df["filter"][aa]]["Eff_centre"].values[0]
        width = PS_filter_df[PS_filter_df['Filter'] == merged_df["filter"][aa]]["Eff_width"].values[0]

        centres.append(centre)
        widths.append(width)

    merged_df["centre"] = centres
    merged_df["width"] = widths

    merged_df = merged_df.replace({'None' : 'NaN'})

    ##################################################################################################
    ##################################################################################################


    new_df["Mag_rounded(AB)"] = new_df["Mag(AB)"].round(decimals=2)
    new_df["eMag_rounded(AB)"] = new_df["eMag(AB)"].round(decimals=2)

    new_df.to_csv(f"{_mkdir('Photometry/csv_files')}/photometry_ZP-corrected_MJD~{mjd_of_interest}.csv", index=False)

    ##################################################################################################

    plotter_fn(new_df, merged_df, mjd_of_interest)
